# NumCompute Quickstart Demo

This notebook demonstrates the NumCompute package end-to-end using only Python and NumPy.

## 1. Imports

In [30]:
import os
import sys
import numpy as np

# Allow notebook to import local package from project root
sys.path.append(os.path.abspath('..'))

from numcompute.io import read_csv
from numcompute.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from numcompute.pipeline import Pipeline
from numcompute.rank import rank, percentile
from numcompute.sort_search import stable_sort, multi_key_sort, topk, quickselect, binary_search
from numcompute.stat import mean, median, std, histogram, quantiles
from numcompute.metrics import accuracy, precision, recall, f1, mse, confusion_matrix
from numcompute.optim import grad, jacobian
from numcompute.benchmarking import run_all_benchmarks
from numcompute.utils import sigmoid, relu, softmax, logsumexp, pairwise_distances, topk_indices, iter_batches


## 2. Read CSV with Missing Values

In [29]:
csv_path = 'sample.csv'

with open(csv_path, 'w') as f:
    f.write('age,study,score\n')
    f.write('18,5,70\n')
    f.write('19,6,80\n')
    f.write('20,,90\n')  # missing value
    f.write('21,8,85\n')

X = read_csv(csv_path, return_dict=False)
X


array([[18.,  5., 70.],
       [19.,  6., 80.],
       [20., nan, 90.],
       [21.,  8., 85.]])

## 3. Preprocessing

In [16]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled


array([[-1.34164079, -1.06904497, -1.52127766],
       [-0.4472136 , -0.26726124, -0.16903085],
       [ 0.4472136 ,         nan,  1.18321596],
       [ 1.34164079,  1.33630621,  0.50709255]])

In [17]:
minmax = MinMaxScaler()
X_minmax = minmax.fit_transform(X)
X_minmax


array([[0.        , 0.        , 0.        ],
       [0.33333333, 0.33333333, 0.5       ],
       [0.66666667,        nan, 1.        ],
       [1.        , 1.        , 0.75      ]])

In [18]:
categories = np.array(['A', 'B', 'A', 'C'])
encoder = OneHotEncoder()
encoder.fit_transform(categories)


array([[1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 0., 1.]])

## 4. Pipeline

In [19]:
pipe = Pipeline([
    ('scale', StandardScaler()),
    ('minmax', MinMaxScaler())
])

pipe.fit_transform(X)


array([[0.        , 0.        , 0.        ],
       [0.33333333, 0.33333333, 0.5       ],
       [0.66666667,        nan, 1.        ],
       [1.        , 1.        , 0.75      ]])

## 5. Statistics

In [20]:
X_clean = np.nan_to_num(X, nan=0.0)

print('Mean:', mean(X_clean)) #Mean is computed over all values in the dataset.
print('Median:', median(X_clean))
print('Std:', std(X_clean))

edges, counts = histogram(X[:, 2], bins=3)
print('Histogram edges:', edges)
print('Histogram counts:', counts)

print('Quantile 0.5:', quantiles(X, 0.5))



Mean: 35.166666666666664
Median: 19.5
Std: 33.46100549727831
Histogram edges: [70.         76.66666667 83.33333333 90.        ]
Histogram counts: [1 1 2]
Quantile 0.5: 20.0


## 6. Ranking and Percentiles

In [21]:
scores = np.array([70, 90, 80, 80])

print('Average rank:', rank(scores, method='average'))
print('Dense rank:', rank(scores, method='dense'))
print('90th percentile:', percentile(scores, 90))


Average rank: [0.  3.  1.5 1.5]
Dense rank: [0 2 1 1]
90th percentile: 87.0


## 7. Sorting and Searching

In [22]:
arr = np.array([10, 50, 20, 40])

print('Stable sort:', stable_sort(arr))
print('Top 2 values:', topk(arr, 2, largest=True, return_indices=False))
print('2nd smallest using quickselect:', quickselect(arr, 1))
print('Binary search for 40:', binary_search(np.array([10, 20, 40, 50]), 40))

table = np.array([
    [2, 3],
    [1, 5],
    [1, 2],
    [2, 1]
])

print('Multi-key sort:')
print(multi_key_sort(table, keys=[0, 1]))


Stable sort: [10 20 40 50]
Top 2 values: [50 40]
2nd smallest using quickselect: 20
Binary search for 40: (2, True)
Multi-key sort:
[[1 2]
 [1 5]
 [2 1]
 [2 3]]


## 8. Metrics

In [23]:
y_true = np.array([1, 0, 1, 1])
y_pred = np.array([1, 0, 0, 1])

print('Accuracy:', accuracy(y_true, y_pred))
print('Precision:', precision(y_true, y_pred))
print('Recall:', recall(y_true, y_pred))
print('F1:', f1(y_true, y_pred))
print('Confusion matrix:')
print(confusion_matrix(y_true, y_pred))
print('MSE:', mse(np.array([1, 2, 3]), np.array([1, 2, 4])))


Accuracy: 0.75
Precision: 1.0
Recall: 0.6666666666666666
F1: 0.8
Confusion matrix:
[[1 0]
 [1 2]]
MSE: 0.3333333333333333


## 9. Finite-Difference Gradient and Jacobian

In [24]:
def f(x):
    return x[0]**2 + x[1]**2

print('Gradient:', grad(f, np.array([3.0, 4.0])))

def F(x):
    return np.array([x[0] + x[1], x[0] * x[1]])

print('Jacobian:')
print(jacobian(F, np.array([2.0, 3.0])))


Gradient: [6. 8.]
Jacobian:
[[1. 1.]
 [3. 2.]]


## 10. Utils

In [25]:
u = np.array([-2.0, 0.0, 2.0])

print('Sigmoid:', sigmoid(u))
print('ReLU:', relu(u))
print('Softmax:', softmax(np.array([1.0, 2.0, 3.0])))
print('LogSumExp:', logsumexp(np.array([1000.0, 1000.0])))

points = np.array([[0, 0], [3, 4]])
print('Pairwise distances:')
print(pairwise_distances(points))
print('Top-k indices:', topk_indices(np.array([10, 50, 20, 40]), 2))
print('Batches:', list(iter_batches(5, 2)))


Sigmoid: [0.11920292 0.5        0.88079708]
ReLU: [0. 0. 2.]
Softmax: [0.09003057 0.24472847 0.66524096]
LogSumExp: 1000.6931471805599
Pairwise distances:
[[0. 5.]
 [5. 0.]]
Top-k indices: [1 3]
Batches: [array([0, 1]), array([2, 3]), array([4])]


## 11. Benchmarking

In [31]:
run_all_benchmarks()


=== Benchmark Results ===

MSE Benchmark:
loop_time: 0.0202752000
vectorised_time: 0.0009021400
speedup: 22.4745615978

Accuracy Benchmark:
loop_time: 0.0117913600
vectorised_time: 0.0002730400
speedup: 43.1854699480
